# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
! pip install -q schedule pytest # установка библиотек, если ещё не

In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
 def get_book_data(book_url: str) -> dict:
    """
    Сбор информации о книге с указанной веб-страницы.
    
    Входные аргументы:
        book_url (str): url страницы книги
    
    Возвращаемые значения:
    
    При успешном запросе:
        book_data (dict): словарь, содержащий:
            title (str): название книги,
            price (str): цена книги,
            ranking (int): рейтинг книги от 1 до 5,
            quantity_in_stock (int): количество экземпляров книги в наличии,
            description (str): описание книги,
            product_information (dict): словарь, содержащий доп.характеристики из таблицы Product Information
    
    При неуспешном запросе:
        сообщение об ошибке и пустой словарь
            
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    response = requests.get(book_url)
    if response.status_code == 200:
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, 'html.parser')
        title = soup.find('h1').get_text().strip()
        price = soup.find('p', attrs={'class': 'price_color'}).get_text()
        ranking = soup.find('p', attrs={'class': 'star-rating'})
        ranking_text = ranking['class'][1]
        ranking_num = {
            'One': 1,
            'Two': 2,
            'Three': 3,
            'Four': 4,
            'Five': 5
        }
        ranking = ranking_num[ranking_text]
        in_stock = soup.find('p', attrs={'class': 'instock availability'}).get_text().strip()
        if 'In stock' in in_stock:
            quantity_text = in_stock.split('(')[1]
            quantity_text = quantity_text.split()[0]
            quantity_in_stock = int(quantity_text)
        else:
            quantity_in_stock = 0
        description = soup.find('div', attrs={'id': 'product_description'})
        if description is not None:
            description = description.find_next('p').get_text().strip()
        else: 
            description = ''
        product_information = {}
        for row in soup.find_all('tr'):
            features_name = row.find('th').get_text()
            features_value = row.find('td').get_text()
            product_information[features_name] = features_value
        book_data = {
            'title': title,
            'price': price,
            'ranking': ranking,
            'quantity_in_stock': quantity_in_stock,
            'description': description,
            'product_information': product_information
        }
        
        return book_data
    
    else:
        print('Ошибка запроса')
        return {}
    
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [4]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'price': '£51.77',
 'ranking': 3,
 'quantity_in_stock': 22,
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who p

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [5]:
def scrape_books(is_save: bool = False):
    """
    Сбор информации обо всех книгах из каталога books.toscrape.com.
    
    Входные аргументы:
        is_save (bool): аргумент-флаг, отвечающий за сохранение результата в файл.
           при True данные сохраняются в файл books_data.txt,
           при False не сохраняется
    
    Возвращаемые значения:
        books_data (list): список словарей с информацией о книгах, полученной с помощью get_book_data
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    books_data = []
    for page in range(1, 51):
        page_url = f'https://books.toscrape.com/catalogue/page-{page}.html'
        response = requests.get(page_url)
        if response.status_code == 200:
            response.encoding = 'utf-8'
            soup = BeautifulSoup(response.text, 'html.parser')
            books_on_page = soup.find_all('h3')
            for book in books_on_page:
                book_link = book.find('a')
                book_href = book_link['href']
                book_url = 'https://books.toscrape.com/catalogue/' + book_href
                book_data = get_book_data(book_url)
                books_data.append(book_data)
    if is_save:
        with open('books_data.txt', 'w', encoding='utf-8') as f:
            f.write(str(books_data))
    return books_data

    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [6]:
# Проверка работоспособности функции
res = scrape_books(is_save=True) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки
print(res[0])

<class 'list'> 1000
{'title': 'A Light in the Attic', 'price': '£51.77', 'ranking': 3, 'quantity_in_stock': 22, 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place t

## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [23]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
schedule.every().day.at("19:00").do(scrape_books, is_save=True)
while True:
    schedule.run_pending()
    time.sleep(60)
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

KeyboardInterrupt: 

## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [7]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! cd .. && python -m pytest tests/test_scraper.py

============================= test session starts ==============================
platform darwin -- Python 3.12.2, pytest-7.4.4, pluggy-1.0.0
rootdir: /Users/eva/Downloads/books_scraper
plugins: anyio-4.2.0
collected 3 items                                                              

tests/test_scraper.py ...                                                [100%]

======================== 3 passed in 703.88s (0:11:43) =========================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ:  Было задано разработать систему для автоматического сбора данных о книгах с сайта Books to Scrape, используя библиотеки requests и BeautifulSoup, настроить регулярную выгрузку через schedule, написать тесты с pytest и оформить проект на GitHub. Все задачи удалось решить, в задаче 1 написан парсер для одной книги, в задаче 2 парсер для всего каталога, данные сохраняются в файл, в задаче 3 настроена выгрузка, в задаче 4 написаны и пройдены 3 теста. Создан проект на GitHub. Задачи на написание самого кода парсинга (Задания 1 и 2) дались относительно легко, хотя и потребовали возвращаться к задаче 1 из-за ошибок (например, когда во время выполнения задачи 2 получалась ошибка из-за того, что не у всех книг есть описание). Работа с терминалом для организации проекта оказались для меня значительно сложнее, так как я раньше вообще не работала с терминалом. `

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Несмотря на то, что задачи 1 и 2 не вызвали больших затруднений, возникли нюансы с кодировкой (пришлось указывать response.encoding, так как цены выводились с символом Â) и с извлечением описания (помог метод find_next). При выполнении задачи 2 возникла ошибка, так как не у всех книг на сайте есть блок с описанием (product description). Пришлось вернуться к задаче 1 и добавить проверку.
Больше всего трудностей возникло при работе с терминалом, так как я почти никогда не делала этого раньше. Пришлось разбираться с навигацией, созданием папок и файлов, перемещением файлов
При запуске указанного кода в задаче 4 возникла ошибка ModuleNotFoundError: No module named 'scraper', хотя в терминале проверила, что все тесты были пройдены. С помощью ИИ исправила это командой ! cd .. && python -m pytest tests/test_scraper.py. В задаче 3 из-за бесконечного цикла while пришлось прервать выполнение, затем перезапускать последовательно ячейки, пропуская этот блок. В задаче 2 долго осуществлялся парсинг, в задаче 4 тесты тоже шли очень долго из-за этого.`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: Использовала ИИ для отдельных проблем, возникавших с кодом. Промпты, которые я использовала:
"Как извлечь текст из следующего тега после определенного элемента в BeautifulSoup?" ИИ подсказал метод find_next, что помогло при парсинге описания книг.
"Почему при парсинге цен на сайте появляется символ Â и как это исправить в requests?" ИИ подсказал, что нужно явно указать response.encoding = 'utf-8'.
"Приведи пример правильного оформления документации для функции на Python по стандарту PEP-8".
Также по поводу задачи 4 спросила ИИ про ошибку ModuleNotFoundError: No module named 'scraper'. ИИ подсказал исправление ! cd .. && python -m pytest tests/test_scraper.py..`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: Осталось непонятным, как можно оптимизировать Задание 2. В моем решении парсинг 1000 книг занимает очень много времени. Можно ли использовать упомянутые в асинхронных материалах на платформе мок-объекты в pytest, чтобы тесты не делали реальные запросы к сайту и проходили быстрее, и как в целом можно оптимизировать парсинг`

**5. Самооценка по критериям задания**:

`Ответ: Задачи 1, 3 и 5 считаю выполненными полностью. Задачи 2 и 4 решены, но есть риски по времени выполнения (парсинг очень долгий, возможно, его нужно было как-то оптимизировать, тесты тоже из-за этого долгие).`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ:  Ценным было то, что получилось сделать небольшой проект на GitHub. Самым сложным оказалась работа с терминалом. Хотелось бы больше материалов по использованию мок-объектов, чтобы сокращать время выполнения тестов, и по оптимизации парсинга.`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

